# Wheat Detection - Google Colab Runner

This notebook runs your existing project code in Colab with minimal changes.
It executes the same experiment commands used in `run_proposal_experiments.ps1`.


## 1) Mount Google Drive

Your outputs and datasets can persist in Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Clone (or refresh) your repo

Set `REPO_URL` to your GitHub repository URL.


In [ ]:
import os

REPO_URL = "https://github.com/2476327115/wheat-detection.git"
WORKDIR = "/content/wheat-detection"

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd


## 3) Install dependencies


In [ ]:
!pip -q install --upgrade pip
!pip -q install pandas matplotlib pillow tqdm

import torch, torchvision, pandas, matplotlib
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())


## 4) Prepare dataset path

Expected structure under `DATA_DIR`:
- `train.csv`
- `train/` (jpg files)

You can point `DATA_DIR` to:
- a Kaggle dataset you downloaded to Colab, or
- a folder in Google Drive.


In [ ]:
from pathlib import Path

# Option A: data in Drive
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/wheat-detection/data")

# Option B example (if you download in Colab):
# DATA_DIR = Path("/content/global-wheat-detection")

assert (DATA_DIR / "train.csv").exists(), f"Missing {(DATA_DIR / 'train.csv')}"
assert (DATA_DIR / "train").exists(), f"Missing {(DATA_DIR / 'train')}"
print("Using DATA_DIR:", DATA_DIR)


## 5) Configure run settings


In [ ]:
from pathlib import Path

EPOCHS = 30
BATCH_SIZE = 4
NUM_WORKERS = 2
EVAL_EVERY = 1
EARLY_STOP_PATIENCE = 3

BASE_OUT = Path("/content/drive/MyDrive/wheat_outputs_colab")
BASE_OUT.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", BASE_OUT)


## 6) Run the 4 proposal experiments


In [ ]:
import sys
from src import faster_rcnn_wheat as train_mod

DATA_DIR_STR = str(DATA_DIR)
BASE_OUT_STR = str(BASE_OUT)

def run_train(args):
    sys.argv = ["faster_rcnn_wheat.py", *args]
    train_mod.main()

# 1) Baseline
run_train([
    "--data-dir", DATA_DIR_STR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--eval-every", str(EVAL_EVERY),
    "--early-stop-patience", str(EARLY_STOP_PATIENCE),
    "--backbone", "resnet50",
    "--experiment-tag", "baseline_r50_default",
    "--output-dir", f"{BASE_OUT_STR}/baseline_r50_default",
])

# 2) Anchor tuning
run_train([
    "--data-dir", DATA_DIR_STR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--eval-every", str(EVAL_EVERY),
    "--early-stop-patience", str(EARLY_STOP_PATIENCE),
    "--backbone", "resnet50",
    "--anchor-sizes", "16,24,32,48,64",
    "--anchor-aspects", "0.5,1.0,2.0",
    "--experiment-tag", "anchor_tuned_r50",
    "--output-dir", f"{BASE_OUT_STR}/anchor_tuned_r50",
])

# 3) Anchor + augmentation (ResNet-50)
run_train([
    "--data-dir", DATA_DIR_STR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--eval-every", str(EVAL_EVERY),
    "--early-stop-patience", str(EARLY_STOP_PATIENCE),
    "--backbone", "resnet50",
    "--anchor-sizes", "16,24,32,48,64",
    "--use-scale-jitter",
    "--use-random-crop",
    "--experiment-tag", "anchor_aug_r50",
    "--output-dir", f"{BASE_OUT_STR}/anchor_aug_r50",
])

# 4) Anchor + augmentation (ResNet-101)
run_train([
    "--data-dir", DATA_DIR_STR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--eval-every", str(EVAL_EVERY),
    "--early-stop-patience", str(EARLY_STOP_PATIENCE),
    "--backbone", "resnet101",
    "--anchor-sizes", "16,24,32,48,64",
    "--use-scale-jitter",
    "--use-random-crop",
    "--experiment-tag", "anchor_aug_r101",
    "--output-dir", f"{BASE_OUT_STR}/anchor_aug_r101",
])



## 7) Summarize experiment results


In [ ]:
!python src/summarize_experiments.py   --base-out "$BASE_OUT_STR"   --save "$BASE_OUT_STR/experiment_summary.csv"


## 8) Quick visualization


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

summary_path = BASE_OUT / "experiment_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))

for exp in ["baseline_r50_default", "anchor_tuned_r50", "anchor_aug_r50", "anchor_aug_r101"]:
    preview = BASE_OUT / exp / "val_prediction_preview.png"
    if preview.exists():
        print(f"\n{exp} preview:")
        display(Image(filename=str(preview)))
